# 🎮 Game of 24 - Tree of Thoughts with CodeAct (Gemini)

> **COMPLETE VERSION - All features from OpenAI notebook**  
> **Optimized for Gemini API Free Tier**  
> **Limits:** 20 requests/min, 14k requests/day  
> **Built-in safety:** 3.5s delays (17 req/min), daily tracking, auto-retry

---

## 🚀 Quick Start

**First time?** Run Cell 1 to create `.env` file  
**Need help?** Check the documentation cells below  

---

## ⚙️ What This Does

1. **Input:** 4 numbers (e.g., `[4, 5, 6, 10]`)
2. **Process:** Tree of Thoughts search with CodeAct pattern
3. **Output:** Mathematical expression that equals 24
4. **Saves:** Complete tree in JSON format

---

## 🎯 Expected Performance

| Puzzle Type | API Calls | Runtime | Success Rate |
|------------|-----------|---------|---------------|
| Easy | 30-50 | 2-3 min | >90% |
| Medium | 80-120 | 5-7 min | ~80% |
| Hard | 150-200 | 10-12 min | ~60% |

**Daily capacity:** ~50-175 puzzles (depending on difficulty)

---

## 📋 Setup Checklist

- [ ] Run Cell 1 below (setup & env configuration)
- [ ] When prompted, paste your Gemini API key
- [ ] Select your preferred model
- [ ] Run cells 2-9 (core setup)
- [ ] Run example cell (first puzzle!)

---

## 💡 Key Features

✅ **Safe Code Execution:** LLM generates Python, runs in sandbox  
✅ **Selective Exhaustive Rescue (SER):** Fallback when LLM proposals are weak  
✅ **Exhaustive First Moves:** Option to try ALL ~24 first moves deterministically  
✅ **Delayed Fraction Preservation (DFP):** Rescue fragile fractional states  
✅ **Hybrid Evaluation:** Heuristics + LLM for robust state scoring  
✅ **Global State Tracking:** Never explore same state twice  
✅ **Rate Limiting:** Automatic 3.5s delays, stays under limits  
✅ **Usage Tracking:** Real-time quota monitoring  
✅ **Error Handling:** Auto-retry on rate limits  
✅ **Complete Logs:** Every thought, code, and observation saved  
✅ **Secure:** API key stored in `.env`, never exposed in notebook  

---

**Let's begin! 👇**

In [1]:
# ============================================================================
# STEP 1: ENVIRONMENT SETUP & API KEY CONFIGURATION
# ============================================================================
# This cell handles .env file creation and API key setup
# RUN THIS FIRST!

import os
from pathlib import Path
import getpass

ENV_FILE = ".env"
WORKSPACE_DIR = Path.cwd()

def setup_env_file():
    """
    Create or verify .env file with Gemini API key and model selection
    """
    env_path = WORKSPACE_DIR / ENV_FILE
    
    # Check if .env file already exists
    if env_path.exists():
        print("✓ .env file found!")
        with open(env_path, 'r') as f:
            content = f.read()
        
        # Verify it has required keys
        if 'GEMINI_API_KEY' in content:
            print("✓ GEMINI_API_KEY already configured")
            
            # Extract and display key status
            for line in content.split('\n'):
                if line.startswith('GEMINI_API_KEY'):
                    key_part = line.split('=')[1][:10] + '***'
                    print(f"  Key starts with: {key_part}")
                    break
        else:
            print("⚠️ GEMINI_API_KEY not found in .env")
            update_env_file(env_path)
    else:
        print("📝 Creating new .env file...")
        create_env_file(env_path)
    
    return env_path

def create_env_file(env_path):
    """
    Create a new .env file with user input
    """
    print("\n" + "="*70)
    print("🔑 GEMINI API KEY SETUP")
    print("="*70)
    print()
    print("To get your Gemini API key:")
    print("1. Visit: https://aistudio.google.com/app/apikey")
    print("2. Click 'Create API key'")
    print("3. Copy the key and paste below")
    print()
    
    api_key = getpass.getpass("Paste your Gemini API key: ").strip()
    
    if not api_key:
        print("❌ No API key provided. Exiting setup.")
        return
    
    # Model selection
    print("\nAvailable models:")
    models = [
        "gemma-4-31b-it",
        "gemma-3-12b-it",
        "gemini-1.5-flash",
    ]
    for i, model in enumerate(models, 1):
        print(f"  {i}. {model}")
    
    choice = input("\nSelect model (1-3, default: 1): ").strip() or "1"
    try:
        model_idx = int(choice) - 1
        if 0 <= model_idx < len(models):
            selected_model = models[model_idx]
        else:
            selected_model = models[0]
            print(f"⚠️ Invalid choice, using default: {selected_model}")
    except:
        selected_model = models[0]
        print(f"⚠️ Invalid input, using default: {selected_model}")
    
    # Write .env file
    env_content = f"""# Gemini API Configuration
# DO NOT COMMIT THIS FILE TO GITHUB!
# Add .env to .gitignore

GEMINI_API_KEY={api_key}
GEMINI_MODEL={selected_model}
"""
    
    with open(env_path, 'w') as f:
        f.write(env_content)
    
    # Verify .gitignore
    gitignore_path = WORKSPACE_DIR / ".gitignore"
    if gitignore_path.exists():
        with open(gitignore_path, 'r') as f:
            gitignore_content = f.read()
        if '.env' not in gitignore_content:
            with open(gitignore_path, 'a') as f:
                f.write("\n.env\n")
            print("✓ Added .env to .gitignore")
    else:
        with open(gitignore_path, 'w') as f:
            f.write(".env\n")
        print("✓ Created .gitignore with .env")
    
    print(f"\n✅ .env file created successfully!")
    print(f"   Model: {selected_model}")
    print(f"   Location: {env_path}")

def update_env_file(env_path):
    """
    Update existing .env file with API key
    """
    print("\n📝 Updating .env file...")
    api_key = getpass.getpass("Paste your Gemini API key: ").strip()
    
    with open(env_path, 'r') as f:
        content = f.read()
    
    # Update or add the key
    if 'GEMINI_API_KEY' in content:
        lines = content.split('\n')
        for i, line in enumerate(lines):
            if line.startswith('GEMINI_API_KEY'):
                lines[i] = f'GEMINI_API_KEY={api_key}'
        content = '\n'.join(lines)
    else:
        content += f'\nGEMINI_API_KEY={api_key}'
    
    with open(env_path, 'w') as f:
        f.write(content)
    
    print("✅ .env file updated!")

# Run setup
env_file = setup_env_file()

# Load environment variables
from dotenv import load_dotenv
load_dotenv(env_file)

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
GEMINI_MODEL = os.getenv('GEMINI_MODEL', 'gemma-4-31b-it')

print(f"\n✅ Environment loaded!")
print(f"   API Key: {GEMINI_API_KEY[:10]}***" if GEMINI_API_KEY else "   API Key: NOT SET")
print(f"   Model: {GEMINI_MODEL}")

✓ .env file found!
✓ GEMINI_API_KEY already configured
  Key starts with: AQ.Ab8RN6L***

✅ Environment loaded!
   API Key: AQ.Ab8RN6L***
   Model: gemma-4-31b-it


In [3]:
# ============================================================================
# STEP 2: IMPORTS & GEMINI API SETUP
# ============================================================================

import re
import json
import sympy
import numpy as np
from datetime import datetime
from typing import List, Dict, Tuple, Any
import itertools
import time
import sys
from io import StringIO
import ast
import math
from itertools import combinations

# Gemini API setup
import google.generativeai as genai
import os

# Configure Gemini API
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.0-flash")

if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    print(f"✓ Gemini API configured")
else:
    print("❌ GEMINI_API_KEY not set. Please run Cell 1 first.")
    raise ValueError("API key not configured")

# Initialize the model
try:
    model = genai.GenerativeModel(GEMINI_MODEL)
    print(f"✓ Model initialized: {GEMINI_MODEL}")
except Exception as e:
    print(f"❌ Error initializing model: {e}")
    raise

# Rate limiting configuration for free tier
# Free tier limits: 20 requests/minute, 14k requests/day
API_DELAY = 0.01  # 1.5 seconds between requests = ~40 requests/minute (safe margin)
last_api_call_time = 0

def rate_limited_api_call():
    """Ensure we don't exceed rate limits"""
    global last_api_call_time
    current_time = time.time()
    time_since_last_call = current_time - last_api_call_time
    
    if time_since_last_call < API_DELAY:
        sleep_time = API_DELAY - time_since_last_call
        # Uncomment to see rate limiting in action:
        # print(f"⏱ Rate limiting: sleeping {sleep_time:.1f}s...", end="", flush=True)
        time.sleep(sleep_time)
        # print(" done")
    
    last_api_call_time = time.time()

def gemini_generate(prompt: str, n: int = 1, temperature: float = 0.7) -> List[str]:
    """
    Generate content using Gemini API with rate limiting
    
    Args:
        prompt: The prompt text
        n: Number of completions to generate
        temperature: Sampling temperature (0.0-2.0)
    
    Returns:
        List of generated text responses
    """
    outputs = []
    
    generation_config = genai.types.GenerationConfig(
        temperature=temperature,
        candidate_count=1,
        max_output_tokens=1500,
    )
    
    for i in range(n):
        try:
            # Apply rate limiting before each API call
            rate_limited_api_call()
            
            response = model.generate_content(
                prompt,
                generation_config=generation_config
            )
            
            text = response.text
            outputs.append(text)
            
        except Exception as e:
            print(f"⚠ Error generating completion {i+1}: {e}")
            # If rate limited, wait longer
            if "429" in str(e) or "quota" in str(e).lower():
                print("⏱ Rate limit hit - waiting 60 seconds...")
                time.sleep(60)
            outputs.append("")
    
    return outputs

print(f"✓ Rate limiting configured: {API_DELAY}s delay (~{int(60/API_DELAY)} req/min)")
print("✓ Gemini API wrapper ready")

✓ Gemini API configured
✓ Model initialized: gemma-4-31b-it
✓ Rate limiting configured: 0.01s delay (~6000 req/min)
✓ Gemini API wrapper ready


In [4]:
# ============================================================================
# STEP 3: SAFE SANDBOX FOR CODE EXECUTION
# ============================================================================

class SafeAgentSandbox:
    """A SECURE sandbox that only allows safe math operations"""
    def __init__(self):
        # ONLY allow these safe built-ins
        self.safe_builtins = {
            'print': print,
            'range': range,
            'len': len,
            'sum': sum,
            'min': min,
            'max': max,
            'abs': abs,
            'round': round,
            'int': int,
            'float': float,
            'str': str,
            'list': list,
        }
        self.globals = {
            "__builtins__": self.safe_builtins,
            "math": __import__("math"),
        }
        
    def run(self, code):
        """Execute code in restricted environment with validation"""
        # Blacklist dangerous operations
        dangerous_keywords = ['import', 'open', 'exec', 'eval', '__', 'os', 'sys', 'subprocess', 'file']
        if any(keyword in code.lower() for keyword in dangerous_keywords):
            return "Error: Code contains forbidden operations", None
        
        # Check for number reuse (e.g., numbers[1] + numbers[1])
        index_pattern = r'numbers\[(\d+)\]'
        indices_used = re.findall(index_pattern, code)
        if len(indices_used) != len(set(indices_used)):
            return "Error: Cannot use the same number twice in one operation", None
        
        # Prevent using arbitrary numbers not in the 'numbers' array
        lines = code.split('\n')
        for line in lines:
            if '#' in line:
                line = line.split('#')[0]
            line = line.strip()
            
            if not line or line.startswith('numbers = ') or line.startswith('remaining = ') or line.startswith('new_numbers = ') or line.startswith('print('):
                continue
                
            if line.startswith('res = ') and any(op in line for op in [' + ', ' - ', ' * ', ' / ', ' % ', ' // ']):
                rhs = line.split('res = ', 1)[1]
                numeric_literals = re.findall(r'(?<!\[)\b(\d+(?:\.\d+)?)\b(?!\])', rhs)
                
                for num_str in numeric_literals:
                    if f'[{num_str}]' in rhs:
                        continue
                    return f"Error: Cannot use arbitrary number {num_str} - must use only numbers from the array via numbers[index]", None
        
        try:
            output_buffer = StringIO()
            sys.stdout = output_buffer
            
            exec(code, self.globals)
            
            sys.stdout = sys.__stdout__
            output = output_buffer.getvalue().strip() or "Code executed successfully."
            
            new_state = self.globals.get('new_numbers', None)
            
            original_count = len(self.globals.get('numbers', []))
            new_count = len(new_state) if new_state else 0
            expected_count = original_count - 1
            
            if new_state and new_count != expected_count:
                return f"Error: Invalid operation - expected {expected_count} numbers, got {new_count}", None
            
            return output, new_state
            
        except Exception as e:
            sys.stdout = sys.__stdout__
            return f"Error: {str(e)}", None

# Initialize global sandbox
sandbox = SafeAgentSandbox()

# Test the sandbox
print("✓ Safe Sandbox initialized")
test_code = """
numbers = [4, 5, 6, 10]
res = numbers[0] + numbers[1]
remaining = numbers[2:]
new_numbers = [res] + remaining
print(new_numbers)
"""
test_output, test_state = sandbox.run(test_code)
print(f"Test result: {test_output}")

✓ Safe Sandbox initialized


In [5]:
# ============================================================================
# STEP 4: PROMPTS FOR CODEACT PATTERN
# ============================================================================

PROPOSE_PROMPT_CODEACT = """<start_of_turn>user
You are solving Game of 24 using executable Python code. Generate MULTIPLE possible next steps.

EXAMPLE - Input: [2, 8, 8, 14]
Possible next steps:

Step 1:
Thought: Add 2 and 8 to get 10
Math: 2 + 8 = 10
Remaining: [10, 8, 14]
Code: ```python
numbers = [2, 8, 8, 14]
res = numbers[0] + numbers[1]  # 2 + 8 = 10
remaining = [numbers[2], numbers[3]]
new_numbers = [res] + remaining
print(new_numbers)
```

Step 2:
Thought: Divide 8 by 2 to get 4
Math: 8 / 2 = 4
Remaining: [4, 8, 14]
Code: ```python
numbers = [2, 8, 8, 14]
res = numbers[1] / numbers[0]  # 8 / 2 = 4
remaining = [numbers[2], numbers[3]]
new_numbers = [res] + remaining
print(new_numbers)
```

Now generate MULTIPLE possible next steps for the current state.

Original puzzle: {original_input}
Steps taken so far:
{history}

Current numbers: {input}

Provide 5-8 DIFFERENT next steps using DIFFERENT operations (+, -, *, /).

CRITICAL REQUIREMENT: For EACH step, you MUST:
1. Thought: Describe the operation
2. Math: CALCULATE the result (e.g., "5 * 6 = 30")
3. Remaining: Show what numbers are left
4. Code: Write the Python code

IMPORTANT:
- Calculate the Math BEFORE writing code
- Show what remains BEFORE writing code
- Ensure diversity - include multiplication, addition, subtraction, division
- 🔧 TRY BOTH ORDERS for multiplication and division
- 🧠 LOOKAHEAD: Think 2-3 steps ahead

⚠️ AVOID DEAD PATTERNS (System learns from failures):
- Creating huge numbers (>50) combined with tiny numbers (<2) - creates intractable gaps
- Results like [64, 1, 4] or [100, 0.5, 3] are dead-ends
- System uses HYBRID learning: static rules + patterns from actual failures
- If you see specific examples below, DEFINITELY avoid them

NOTE: System may add learned patterns from this puzzle below (depth ≥ 2).
If you see "LEARNED DEAD-END PATTERNS", those are CONCRETE evidence from actual failures.
- Large ratios between numbers make it harder to reach 24
- Prioritize operations that keep numbers in the 1-30 range

Format for each step:
Thought: [description]
Math: [calculation with result]
Remaining: [list of numbers left]
Code: ```python
[your code here]
```

Possible next steps:

<end_of_turn>
<start_of_turn>model
"""

VALUE_PROMPT_CODEACT = """<start_of_turn>user
Numbers: {input}
Target: 24

ANSWER WITH ONE WORD ONLY - NO EXPLANATION, NO WORKING, NO BULLET POINTS:
Answer "sure" if reachable. Answer "likely" if closest is 20-24. Answer "impossible" otherwise.

Your word:
<end_of_turn>
<start_of_turn>model
"""

# ============================================================================
# CUSTOM TWO-NUMBER PROMPT (NEW)
# ============================================================================
# For terminal 2-number states: pick ONE operation that works best

TWO_NUMBER_PROPOSE_PROMPT = """<start_of_turn>user
FINAL STEP: You have exactly 2 numbers. Pick ONE operation that gets closest to 24.

Original puzzle: {original_input}
Path so far:
{history}

Current numbers: {input}
Let a = {a}, b = {b}

Evaluate all 6 possible operations:
1. a + b = {a} + {b} = {sum_result}
2. a - b = {a} - {b} = {diff_ab}
3. b - a = {b} - {a} = {diff_ba}
4. a * b = {a} * {b} = {prod_result}
5. a / b = {a} / {b} = {div_ab}
6. b / a = {b} / {a} = {div_ba}

PICK THE OPERATION that gets CLOSEST to 24 (even if not exact).
ALWAYS return ONE operation - no matter which is best.

Output format:
Thought: [Which operation gets closest to 24? Why?]
Selected operation: [ONLY ONE: a+b OR a-b OR b-a OR a*b OR a/b OR b/a]
Result: [the number this produces]

<end_of_turn>
<start_of_turn>model
"""

print("✓ CodeAct prompts loaded successfully (including TWO_NUMBER_PROPOSE_PROMPT)")

In [6]:
# ============================================================================
# STEP 5: TREE NODE CLASS & HELPER FUNCTIONS
# ============================================================================

class TreeNode:
    """Represents a node in the Game of 24 search tree"""
    node_counter = 0
    
    def __init__(self, state: List[float], parent=None, action: str = "", value: float = 0.0, depth: int = 0):
        self.id = TreeNode.node_counter
        TreeNode.node_counter += 1
        
        self.state = state
        self.parent = parent
        self.action = action
        self.value = value
        self.depth = depth
        self.children = {}
        self.is_solution = False
        self.is_pruned = False
        self.evaluation = None  # Store evaluation record for distillation
        
        # CodeAct specific
        self.thought = ""
        self.code = ""
        self.observation = ""
        self.path_history = ""
    
    def to_dict(self):
        """Convert node to dictionary for JSON serialization"""
        return {
            'id': self.id,
            'parent_id': self.parent.id if self.parent else None,
            'state': self.state,
            'action': self.action,
            'value': self.value,
            'depth': self.depth,
            'is_solution': self.is_solution,
            'is_pruned': self.is_pruned,
            'codeact': {
                'thought': self.thought,
                'code': self.code,
                'observation': self.observation
            },
            'path_history': self.path_history,
            'evaluation': self.evaluation,
            'num_children': len(self.children),
            'children': {child_id: child.to_dict() for child_id, child in self.children.items()}
        }

def check_solution(state: List[float]) -> Tuple[bool, str]:
    """Check if any single element equals 24"""
    if len(state) == 1:
        if abs(state[0] - 24.0) < 1e-6:
            return True, f"{state[0]} = 24 ✓"
    return False, ""

def state_signature(nums: List[float]) -> tuple:
    """
    Classify state for Delayed Fraction Preservation (DFP).
    
    Returns:
        (non_int_count, small_int_count) tuple
    """
    non_int = [x for x in nums if abs(x - round(x)) > 1e-6]
    small_int = [x for x in nums if abs(x - round(x)) < 1e-6 and x <= 6]
    return len(non_int), len(small_int)

print("✓ TreeNode and helper functions defined")

# ============================================================================
# STEP 5.5: DEAD-END MEMORY (TIM Idea #7 Integration)
# ============================================================================

class DeadEndMemory:
    """
    Learns from pruned/dead-end states to avoid re-exploring similar patterns.
    Implements the "Forget" operation from TIM (Think-in-Memory) paper.
    
    Uses pattern matching on state features:
    - max_value: Maximum number in state
    - min_value: Minimum absolute value in state  
    - range_ratio: max/min ratio (measures spread)
    - has_huge_numbers: Any number > 50?
    - has_tiny_numbers: Any number < 2?
    - num_count: How many numbers remain
    """
    
    def __init__(self, similarity_threshold=0.5):
        self.patterns = []  # List of pattern dicts
        self.similarity_threshold = similarity_threshold
        self.total_skipped = 0
        self.total_checked = 0
        
    def record_dead_end(self, node, reason=""):
        """Extract and store pattern from a dead-end node"""
        pattern = self.extract_pattern(node.state)
        if pattern is not None:
            pattern['reason'] = reason
            pattern['found_at_depth'] = node.depth
            pattern['value_score'] = node.value
            self.patterns.append(pattern)
    
    def extract_pattern(self, state):
        """Extract salient features from a state for pattern matching"""
        if not state or len(state) == 0:
            return None
        
        abs_state = [abs(x) for x in state]
        max_val = max(abs_state)
        # Get minimum non-zero value, or use max_val if all are zero (edge case)
        non_zero_vals = [abs(x) for x in abs_state if abs(x) > 1e-9]
        min_val = min(non_zero_vals) if non_zero_vals else max_val
        
        return {
            'max_value': max_val,
            'min_value': min_val,
            'range_ratio': max_val / min_val if min_val != 0 else float('inf'),
            'has_huge_numbers': any(abs(n) > 50 for n in state),
            'has_tiny_numbers': any(abs(n) < 2 and abs(n) > 0 for n in state),
            'num_count': len(state),
            'sum': sum(abs(n) for n in state)
        }
    
    def is_potential_dead_end(self, new_state):
        """
        Check if new_state matches any known dead-end pattern.
        
        Returns: (is_dead_end: bool, matched_pattern: dict or None)
        """
        self.total_checked += 1
        
        if not new_state or len(new_state) == 0:
            return False, None
        
        new_pattern = self.extract_pattern(new_state)
        if new_pattern is None:
            return False, None
        
        for stored_pattern in self.patterns:
            if self._patterns_similar(new_pattern, stored_pattern):
                self.total_skipped += 1
                return True, stored_pattern
        
        return False, None
    
    def _patterns_similar(self, pattern1, pattern2):
        """
        Check if two patterns are similar using threshold matching.
        
        Similarity checks:
        - Range ratios within threshold of each other
        - Both have or both don't have huge numbers
        - Both have or both don't have tiny numbers
        - Number counts match or are very close
        """
        # Check max values are in similar range
        max_ratio = pattern1['max_value'] / (pattern2['max_value'] + 1e-6)
        if max_ratio < (1 - self.similarity_threshold) or max_ratio > (1 + self.similarity_threshold):
            return False
        
        # Check min values are in similar range (more lenient)
        min_ratio = pattern1['min_value'] / (pattern2['min_value'] + 1e-6)
        if min_ratio < 0.3 or min_ratio > 3:
            return False
        
        # Check huge/tiny number presence
        if pattern1['has_huge_numbers'] != pattern2['has_huge_numbers']:
            return False
        if pattern1['has_tiny_numbers'] != pattern2['has_tiny_numbers']:
            return False
        
        # Check number count is close
        if abs(pattern1['num_count'] - pattern2['num_count']) > 1:
            return False
        
        return True
    
    def get_stats(self):
        """Return statistics about dead-end memory"""
        return {
            'patterns_stored': len(self.patterns),
            'total_checked': self.total_checked,
            'total_skipped': self.total_skipped,
            'skip_rate': f"{100*self.total_skipped/max(1, self.total_checked):.1f}%"
        }
    
    def get_pattern_summary(self, max_patterns=3):
        """
        Generate a human-readable summary of learned dead-end patterns for LLM prompt.
        Only returns patterns that are meaningful (depth >= 2).
        
        Returns:
            str: Formatted pattern summary for inclusion in proposal prompt
        """
        if not self.patterns:
            return ""
        
        # Filter patterns from meaningful depths (depth >= 2)
        meaningful_patterns = [p for p in self.patterns if p.get('found_at_depth', 0) >= 2]
        
        if not meaningful_patterns:
            return ""
        
        # Sort by value score (worst first) and take top patterns
        sorted_patterns = sorted(meaningful_patterns, key=lambda p: p.get('value_score', 1.0))[:max_patterns]
        
        summary_lines = ["LEARNED DEAD-END PATTERNS FROM THIS PUZZLE:"]
        
        for i, pattern in enumerate(sorted_patterns, 1):
            max_val = pattern.get('max_value', '?')
            min_val = pattern.get('min_value', '?')
            ratio = pattern.get('range_ratio', '?')
            has_huge = pattern.get('has_huge_numbers', False)
            has_tiny = pattern.get('has_tiny_numbers', False)
            reason = pattern.get('reason', 'unknown failure')
            depth = pattern.get('found_at_depth', '?')
            
            features = []
            if has_huge:
                features.append(f"huge numbers (>{50})")
            if has_tiny:
                features.append(f"tiny numbers (<2)")
            if ratio != '?' and isinstance(ratio, (int, float)) and ratio > 100:
                features.append(f"extreme range ratio ({ratio:.0f}x)")
            
            feature_str = ", ".join(features) if features else "imbalanced numbers"
            
            summary_lines.append(f"\nPattern {i} (Depth {depth}): [{max_val}, {min_val}, ...] with {feature_str}")
            summary_lines.append(f"  → Result: DEAD-END (reason: {reason})")
            summary_lines.append(f"  → AVOID: Similar combinations of huge/tiny or high ratios")
        
        return "\n".join(summary_lines)

print("✓ DeadEndMemory class (TIM Idea #7) loaded")

# ============================================================================
# STEP 5.6: INCONSISTENCY DETECTION (TIM Idea #4)
# ============================================================================

class InconsistencyDetector:
    """
    Detects when the same game state receives inconsistent evaluation scores.
    
    TiM Idea #4: If same state gets very different scores on re-evaluation,
    the LLM evaluation is unreliable.
    
    Tracks all evaluations and flags states with high variance.
    """
    
    def __init__(self, inconsistency_threshold=0.3):
        """
        Args:
            inconsistency_threshold: If max_score - min_score > threshold,
                                     flag as inconsistent (0-1 scale)
        """
        self.evaluation_history = {}  # state_tuple → list of scores
        self.inconsistency_threshold = inconsistency_threshold
        self.inconsistent_states = []
        self.total_evaluations = 0
        self.re_evaluations = 0
    
    def record_evaluation(self, state, score, depth=0, reasoning=""):
        """Record an evaluation of a state"""
        self.total_evaluations += 1
        
        # Create a hashable key (sorted tuple)
        state_key = tuple(sorted([round(x, 6) for x in state]))
        
        if state_key not in self.evaluation_history:
            self.evaluation_history[state_key] = []
        
        self.evaluation_history[state_key].append({
            'score': score,
            'depth': depth,
            'reasoning': reasoning,
            'timestamp': datetime.now()
        })
        
        # Check for inconsistency if we've seen this state before
        if len(self.evaluation_history[state_key]) > 1:
            self.re_evaluations += 1
            return self._check_inconsistency(state_key)
        
        return None
    
    def _check_inconsistency(self, state_key):
        """Check if evaluations of this state are inconsistent"""
        evaluations = self.evaluation_history[state_key]
        scores = [e['score'] for e in evaluations]
        
        min_score = min(scores)
        max_score = max(scores)
        variance = max_score - min_score
        
        # Flag if variance exceeds threshold
        if variance > self.inconsistency_threshold:
            info = {
                'state': list(state_key),
                'is_inconsistent': True,
                'num_evaluations': len(evaluations),
                'scores': scores,
                'min_score': min_score,
                'max_score': max_score,
                'variance': variance,
                'depths': [e['depth'] for e in evaluations]
            }
            self.inconsistent_states.append(info)
            return info
        
        return {
            'state': list(state_key),
            'is_inconsistent': False,
            'num_evaluations': len(evaluations),
            'scores': scores,
            'variance': variance
        }
    
    def get_inconsistent_states(self):
        """Get list of all states with inconsistent evaluations"""
        return self.inconsistent_states
    
    def get_suspicious_states(self, min_variance=0.15):
        """Get states with moderate variance (suspicious but not flagged)"""
        suspicious = []
        for state_key, evals in self.evaluation_history.items():
            if len(evals) > 1:
                scores = [e['score'] for e in evals]
                variance = max(scores) - min(scores)
                
                if min_variance <= variance <= self.inconsistency_threshold:
                    suspicious.append({
                        'state': list(state_key),
                        'num_evaluations': len(evals),
                        'scores': scores,
                        'variance': variance
                    })
        
        return suspicious
    
    def get_stats(self):
        """Return statistics about evaluations"""
        return {
            'total_evaluations': self.total_evaluations,
            're_evaluations': self.re_evaluations,
            'unique_states': len(self.evaluation_history),
            'inconsistent_states': len(self.inconsistent_states),
            're_evaluation_rate': f"{100*self.re_evaluations/max(1, self.total_evaluations):.1f}%" if self.total_evaluations > 0 else "0%",
            'inconsistency_rate': f"{100*len(self.inconsistent_states)/max(1, self.re_evaluations):.1f}%" if self.re_evaluations > 0 else "0%"
        }

print("✓ InconsistencyDetector class (TIM Idea #4) loaded")


In [7]:
# ============================================================================
# STEP 6: COMPLETE GAME24 SOLVER CLASS (WITH ALL FEATURES)
# ============================================================================

class Game24TreeOfThoughts:
    """Complete Tree of Thoughts solver with ALL features from OpenAI version"""

    def __init__(self, 
                 temperature: float = 0.7,
                 n_evaluate_sample: int = 3,
                 n_select_sample: int = 15,
                 max_steps: int = 6,
                 api_delay: float = 3.5,
                 selection_method: str = 'greedy',
                 exhaustive_depth1: bool = False,
                 enable_ser: bool = False,
                 enable_deadend_memory: bool = True):
        """Initialize the ToT solver with CodeAct and optional Dead-End Memory"""
        self.temperature = temperature
        self.n_evaluate_sample = n_evaluate_sample
        self.n_select_sample = n_select_sample
        self.max_steps = max_steps
        self.api_delay = api_delay
        self.selection_method = selection_method
        self.exhaustive_depth1 = exhaustive_depth1
        self.enable_ser = enable_ser
        self.enable_deadend_memory = enable_deadend_memory
        self.value_cache = {}

        # Tree structure
        self.root = None
        self.all_nodes = []
        self.solutions = []

        # Dead-End Memory (TIM Idea #7)
        self.dead_end_memory = DeadEndMemory(similarity_threshold=0.5) if enable_deadend_memory else None

        # Inconsistency Detector (TIM Idea #4)
        self.inconsistency_detector = InconsistencyDetector(inconsistency_threshold=0.3)

        # Statistics
        self.stats = {
            'total_nodes': 0,
            'api_calls': 0,
            'cache_hits': 0,
            'solutions_found': 0,
            'code_executions': 0,
            'code_errors': 0,
            'daily_requests': 0,
            'session_start': datetime.now(),
            'deadend_memory_enabled': enable_deadend_memory,
            'deadend_memory_skipped': 0
        }

        # Free tier limits
        self.DAILY_LIMIT = 14000
        self.MINUTE_LIMIT = 20

        print(f"✓ Solver initialized")
        print(f"  • Temperature: {temperature}")
        print(f"  • Beam width: {n_select_sample}")
        print(f"  • Max steps: {max_steps}")
        print(f"  • Dead-End Memory: {'ENABLED ✓' if enable_deadend_memory else 'DISABLED'}")
        print(f"  • Inconsistency Detection (TIM Idea #4): ENABLED ✓")
        if exhaustive_depth1:
            print(f"  • Mode: EXHAUSTIVE DEPTH-1")
        if enable_ser:
            print(f"  • Mode: SER ENABLED")

    def check_rate_limits(self):
        """Check if we're approaching rate limits"""
        if self.stats['daily_requests'] >= self.DAILY_LIMIT * 0.9:
            print(f"⚠ WARNING: Approaching daily limit ({self.stats['daily_requests']}/{self.DAILY_LIMIT})")

    def generate_all_first_moves(self, numbers: List[float]) -> List[Dict]:
        """Generate ALL possible first moves exhaustively (~24 moves)"""
        proposals = []
        operations = [
            ('+', lambda a, b: a + b, "{} + {} = {}"),
            ('-', lambda a, b: a - b, "{} - {} = {}"),
            ('*', lambda a, b: a * b, "{} × {} = {}"),
            ('/', lambda a, b: a / b if b != 0 else None, "{} ÷ {} = {}")
        ]
        
        print(f"  🔬 Exhaustively generating all first moves from {numbers}...")
        
        for i, j in combinations(range(len(numbers)), 2):
            a, b = numbers[i], numbers[j]
            remaining_indices = [k for k in range(len(numbers)) if k not in [i, j]]
            remaining = [numbers[k] for k in remaining_indices]
            
            for op_symbol, op_func, math_template in operations:
                for num1, num2, idx1, idx2 in [(a, b, i, j), (b, a, j, i)]:
                    if op_symbol in ['+', '*'] and (num1, num2) != (a, b):
                        continue
                    
                    result = op_func(num1, num2)
                    if result is None or not math.isfinite(result):
                        continue
                    
                    new_state = [result] + remaining
                    math_str = math_template.format(num1, num2, result)
                    thought = f"Exhaustive: {math_str}"
                    
                    code = f"""numbers = {numbers}
res = numbers[{idx1}] {op_symbol} numbers[{idx2}]
remaining = {remaining}
new_numbers = [res] + remaining
print(new_numbers)"""
                    
                    proposal = {
                        'thought': thought,
                        'code': code,
                        'observation': str(new_state),
                        'new_state': new_state,
                        'action': str(new_state)
                    }
                    proposals.append(proposal)
        
        print(f"  ✓ Generated {len(proposals)} exhaustive first moves")
        return proposals
    
    def passes_basic_heuristics(self, move: Dict) -> bool:
        """
        Quick heuristic check to filter obviously bad moves.
        Used in hybrid SER to identify promising moves for LLM evaluation.
        """
        new_state = move['new_state']
        
        # Filter out moves that create problematic states
        if len(new_state) != 3:
            return True  # Keep depth 1 moves
        
        # Don't evaluate moves that create extreme imbalances at depth 1
        max_val = max(abs(x) for x in new_state)
        non_zero = [abs(x) for x in new_state if abs(x) > 1e-9]
        min_val = min(non_zero) if non_zero else max_val
        
        # Skip obviously problematic ratios (>1000x)
        if max_val / min_val > 1000:
            return False
        
        # Skip if creates huge numbers (>500)
        if max_val > 500:
            return False
        
        # Skip if creates tiny fractions (<0.01)
        if any(0 < abs(x) < 0.01 for x in new_state):
            return False
        
        return True
    
    def heuristic_score_move(self, move: Dict) -> float:
        """
        Heuristic scoring for moves when LLM evaluation is skipped.
        Based on distance to 24 and balance.
        """
        new_state = move['new_state']
        
        if len(new_state) != 3:
            return 1.0
        
        # Score based on how close sum/product are to 24
        sum_val = sum(new_state)
        try:
            prod_val = abs(new_state[0] * new_state[1] * new_state[2])
        except:
            prod_val = float('inf')
        
        sum_dist = abs(sum_val - 24)
        prod_dist = abs(prod_val - 24) if prod_val < 10000 else 10000
        
        # Closer to 24 gets higher score
        sum_score = 1.0 / (1.0 + sum_dist)
        prod_score = 1.0 / (1.0 + prod_dist)
        
        return max(sum_score, prod_score)
    
    def propose(self, current_numbers: List[int], original_input: List[int], history: str = "", n_proposals: int = 5, current_depth: int = 0) -> List[Dict]:
        """Generate multiple next state proposals using Gemini with hybrid dead-end awareness"""
        
        # DISPATCH: For 2-number states, use custom single-operation proposal
        if len(current_numbers) == 2:
            print(f"      [DEBUG] 2-number state detected: {current_numbers}, routing to custom propose_two_number()")
            return self.propose_two_number(current_numbers, original_input, history)
        
        proposals = []
        seen_states = set()
        
        prompt = PROPOSE_PROMPT_CODEACT.format(
            original_input=original_input,
            history=history if history else "(Starting state)",
            input=current_numbers
        )
        
        # HYBRID APPROACH: Add dynamic dead-end patterns starting at depth 1
        # (Depth 1→2 is where most proposals happen, so start earlier for better filtering)
        if self.dead_end_memory is not None and current_depth >= 1:
            pattern_summary = self.dead_end_memory.get_pattern_summary(max_patterns=3)
            if pattern_summary:
                prompt += f"\n\n{pattern_summary}\n\nGiven these learned failures, generate proposals that AVOID these dead-end patterns."
                print(f"      [HYBRID] Added {len(self.dead_end_memory.patterns)} learned patterns to prompt")
        
        self.check_rate_limits()
        time.sleep(self.api_delay)
        self.stats['api_calls'] += 1
        self.stats['daily_requests'] += 1
        
        print(f"      [DEBUG] Calling gemini_generate...")
        try:
            response = gemini_generate(prompt, n=1, temperature=self.temperature)[0]
            print(f"      [DEBUG] Got response (length: {len(response)})")
            if not response:
                print(f"      [DEBUG] WARNING: Response is empty!")
                return []
        except Exception as e:
            print(f"      [DEBUG] ⚠ Error generating proposals: {e}")
            return []
        
        # Parse response
        full_pattern = r"Thought:\s*(.+?)\s*Math:\s*(.+?)\s*Remaining:\s*(.+?)(?:Code:.*?```python\n(.*?)\n```)"
        matches = re.findall(full_pattern, response, re.DOTALL | re.IGNORECASE)
        
        print(f"      [DEBUG] Full pattern matches: {len(matches)}")
        
        if not matches:
            thought_code_pattern = r"Thought:\s*(.+?)(?:Code:.*?```python\n(.*?)\n```)"
            old_matches = re.findall(thought_code_pattern, response, re.DOTALL | re.IGNORECASE)
            print(f"      [DEBUG] Fallback pattern matches: {len(old_matches)}")
            matches = [(m[0], "", "", m[1]) for m in old_matches]
        
        # Process each parsed step
        for i, match in enumerate(matches):
            if len(proposals) >= n_proposals:
                break
            
            thought = match[0].strip()
            code = match[3].strip() if len(match) > 3 else ""
            
            if not code:
                print(f"      [DEBUG] Match {i}: no code found")
                continue
            
            print(f"      [DEBUG] Processing match {i}: {thought[:40]}...")
            
            try:
                self.stats['code_executions'] += 1
                sandbox.globals['numbers'] = list(current_numbers)
                observation, new_state = sandbox.run(code)
                
                if new_state and not observation.startswith("Error:"):
                    state_tuple = tuple(sorted(new_state))
                    
                    if state_tuple not in seen_states:
                        proposals.append({
                            'thought': thought,
                            'code': code,
                            'observation': observation,
                            'new_state': new_state,
                            'action': observation
                        })
                        seen_states.add(state_tuple)
                        print(f"      [DEBUG]   ✓ Valid proposal accepted")
                else:
                    print(f"      [DEBUG]   ✗ Rejected: {observation[:50]}")
            except Exception as e:
                print(f"      [DEBUG]   ✗ Exception: {e}")
                continue
        
        print(f"      [DEBUG] Total proposals generated: {len(proposals)}")
        return proposals
    
    def propose_two_number(self, current_numbers: List, original_input: str, history: str = "") -> List[dict]:
        """CUSTOM PROPOSAL FOR 2-NUMBER TERMINAL STATES: ALWAYS create ONE child with best operation"""
        proposals = []
        
        if len(current_numbers) != 2:
            return proposals
        
        a, b = current_numbers[0], current_numbers[1]
        
        # Pre-calculate all 6 possible operations
        sum_result = a + b
        diff_ab = a - b
        diff_ba = b - a
        prod_result = a * b
        
        # Handle division carefully
        try:
            div_ab = a / b if b != 0 else None
        except:
            div_ab = None
        
        try:
            div_ba = b / a if a != 0 else None
        except:
            div_ba = None
        
        # Format pre-computed results for prompt
        div_ab_str = f"{div_ab:.10g}" if div_ab is not None else "undefined (cannot divide by zero)"
        div_ba_str = f"{div_ba:.10g}" if div_ba is not None else "undefined (cannot divide by zero)"
        
        # Format the custom prompt
        prompt_content = TWO_NUMBER_PROPOSE_PROMPT.format(
            original_input=original_input,
            history=history if history else "No prior steps",
            input=f"Current state: {current_numbers}",
            a=a,
            b=b,
            sum_result=sum_result,
            diff_ab=diff_ab,
            diff_ba=diff_ba,
            prod_result=prod_result,
            div_ab=div_ab_str,
            div_ba=div_ba_str
        )
        
        operation_result = None
        try:
            print(f"      [DEBUG] Calling LLM for 2-number state: {current_numbers}")
            response = gemini_generate(prompt_content, n=1, temperature=self.temperature)[0]
            
            # Parse response for single operation
            # Look for patterns like "Selected operation: a + b" or "Operation: a + b"
            operation_patterns = [
                r"Selected operation:\s*(.+?)(?:\n|$)",
                r"Operation:\s*(.+?)(?:\n|$)",
                r"Best operation:\s*(.+?)(?:\n|$)",
                r"Recommended:\s*(.+?)(?:\n|$)"
            ]
            
            for pattern in operation_patterns:
                match = re.search(pattern, response, re.IGNORECASE)
                if match:
                    operation_result = match.group(1).strip()
                    break
            
            if operation_result:
                print(f"      [DEBUG] Parsed operation from LLM: {operation_result}")
        except Exception as e:
            print(f"      [DEBUG] ⚠ Error calling LLM for 2-number: {e}")
        
        # FALLBACK: If LLM failed or didn't return operation, pick best operation ourselves
        if not operation_result:
            print(f"      [DEBUG] LLM didn't return operation, using fallback: a + b")
            operation_result = f"{a} + {b}"  # Default to addition (safest)
        
        print(f"      [DEBUG] Final operation for {current_numbers}: {operation_result}")
        
        # Build code to execute the operation
        code = f"numbers = {list(current_numbers)}\n"
        code += f"a, b = {a}, {b}\n"
        code += f"result = {operation_result}\n"
        code += "new_numbers = [x for x in numbers if x not in [a, b]] + [result]\n"
        code += "new_numbers.sort()\n"
        
        try:
            self.stats['code_executions'] += 1
            sandbox.globals['numbers'] = list(current_numbers)
            observation, new_state = sandbox.run(code)
            
            if new_state and not observation.startswith("Error:"):
                proposals.append({
                    'thought': f"For 2-number state {current_numbers}, operation: {operation_result}",
                    'code': code,
                    'observation': observation,
                    'new_state': new_state,
                    'action': observation
                })
                print(f"      [DEBUG] ✓ 2-number child created: {new_state}")
            else:
                # Even if execution failed, create child with fallback operation
                print(f"      [DEBUG] ⚠ Execution failed: {observation}, trying fallback: a+b")
                operation_result = f"{a} + {b}"
                code = f"numbers = {list(current_numbers)}\n"
                code += f"a, b = {a}, {b}\n"
                code += f"result = {operation_result}\n"
                code += "new_numbers = [x for x in numbers if x not in [a, b]] + [result]\n"
                code += "new_numbers.sort()\n"
                sandbox.globals['numbers'] = list(current_numbers)
                observation, new_state = sandbox.run(code)
                if new_state:
                    proposals.append({
                        'thought': f"2-number state {current_numbers} (fallback): {operation_result}",
                        'code': code,
                        'observation': observation,
                        'new_state': new_state,
                        'action': observation
                    })
                    print(f"      [DEBUG] ✓ 2-number child created with fallback: {new_state}")
        except Exception as e:
            print(f"      [DEBUG] ✗ Exception in 2-number proposal: {e}")
        
        return proposals
    
    def evaluate_state(self, numbers: List[int], is_final: bool = False, force_llm: bool = False) -> Tuple[float, dict]:
        """HYBRID EVALUATION: Heuristics + LLM"""
        eval_record = {
            "state": str(numbers),
            "is_final": is_final,
            "heuristic_checks": {},
            "llm_judgments": [],
            "reasoning": [],
            "score_breakdown": {},
            "final_value": 0.0
        }
        
        # 1. Check if final answer
        if len(numbers) == 1:
            eval_record["heuristic_checks"]["is_single_number"] = True
            if abs(numbers[0] - 24) < 0.001:
                eval_record["heuristic_checks"]["is_solution"] = True
                eval_record["reasoning"].append("✅ SOLUTION: Final state equals 24 - PUZZLE SOLVED!")
                eval_record["final_value"] = 1.0
                return 1.0, eval_record
            else:
                eval_record["heuristic_checks"]["is_solution"] = False
                eval_record["reasoning"].append(f"❌ WRONG ANSWER: Final state is {numbers[0]}, not 24")
                eval_record["final_value"] = 0.001
                return 0.001, eval_record
        
        # 2. Penalize premature 24
        if 24 in numbers or 24.0 in numbers:
            eval_record["heuristic_checks"]["has_premature_24"] = True
            eval_record["reasoning"].append("⚠️ DEAD-END: Contains 24 but not in final state - dead-end trap!")
            eval_record["final_value"] = 0.01
            return 0.01, eval_record
        else:
            eval_record["heuristic_checks"]["has_premature_24"] = False
        
        # 3. Penalize huge numbers
        max_abs = max(abs(n) for n in numbers)
        eval_record["heuristic_checks"]["max_abs_value"] = max_abs
        if max_abs > 1000:
            eval_record["heuristic_checks"]["has_huge_numbers"] = True
            eval_record["reasoning"].append(f"⚠️ HUGE NUMBERS: Max value {max_abs} exceeds 1000 - unlikely to reach 24")
            eval_record["final_value"] = 0.1
            return 0.1, eval_record
        
        # 4. Hard-coded 2-number check (+ optional LLM + soft clamp)
        if len(numbers) == 2:
            a, b = numbers[0], numbers[1]
            operations_to_24 = [
                (abs(a + b - 24) < 0.001, f"{a} + {b} = {a+b}"),
                (abs(a - b - 24) < 0.001, f"{a} - {b} = {a-b}"),
                (abs(b - a - 24) < 0.001, f"{b} - {a} = {b-a}"),
                (abs(a * b - 24) < 0.001, f"{a} * {b} = {a*b}"),
                (abs(a / b - 24) < 0.001 if b != 0 else False, f"{a} / {b} = {a/b if b != 0 else 'undefined'}"),
                (abs(b / a - 24) < 0.001 if a != 0 else False, f"{b} / {a} = {b/a if a != 0 else 'undefined'}")
            ]
            
            reaching_24 = any(op for op, _result in operations_to_24)
            eval_record["heuristic_checks"]["two_number_reaches_24"] = bool(reaching_24)
            
            if reaching_24:
                matching_ops = [result for op, result in operations_to_24 if op]
                eval_record["reasoning"].append(f"✅ 2-NUMBER SOLUTION: Can reach 24! {matching_ops}")
            else:
                eval_record["reasoning"].append(f"❌ 2-NUMBER BLOCKED: No operation between {a} and {b} reaches 24")
            
            # Default (cheap) heuristic-only score unless explicitly forcing an LLM call
            if not force_llm:
                base_value = 1.0 if reaching_24 else 0.001
                eval_record["final_value"] = base_value
                return base_value, eval_record
            
            # LLM scoring for 2-number states + SOFT CLAMP (reachable/non-reachable buckets don't overlap)
            numbers_str = str(numbers)
            prompt = VALUE_PROMPT_CODEACT.format(input=numbers_str)
            
            if prompt in self.value_cache:
                self.stats['cache_hits'] += 1
                cached_value = self.value_cache[prompt]
                eval_record["from_cache"] = True
                eval_record["final_value"] = cached_value
                return cached_value, eval_record
            
            self.check_rate_limits()
            time.sleep(self.api_delay)
            self.stats['api_calls'] += self.n_evaluate_sample
            self.stats['daily_requests'] += self.n_evaluate_sample
            
            value_outputs = []
            for i in range(self.n_evaluate_sample):
                try:
                    response = gemini_generate(prompt, n=1, temperature=0.0)[0]
                    value_outputs.append(response.strip().lower())
                except:
                    value_outputs.append("likely")
            
            value_map = {'impossible': 0.001, 'likely': 0.6, 'sure': 1.0}
            value_names = []
            for output in value_outputs:
                last_word = output.strip().split()[-1] if output.strip() else "likely"
                value_names.append(last_word)
            
            eval_record["llm_judgments"] = value_outputs
            eval_record["reasoning"].append(f"LLM evaluations: {value_names}")
            eval_record["reasoning"].append(
                f"Judgment votes: sure={value_names.count('sure')}, likely={value_names.count('likely')}, impossible={value_names.count('impossible')}"
            )
            
            llm_raw_value = (
                sum(value_map.get(name, 0.6) for name in value_names) / len(value_names)
                if value_names else 0.6
            )
            
            # Soft clamp buckets:
            # reachable -> [0.95, 1.0], blocked -> [0.001, 0.05]
            if reaching_24:
                clamped_value = 0.95 + 0.05 * llm_raw_value
            else:
                clamped_value = 0.001 + 0.049 * llm_raw_value
            
            eval_record["reasoning"].append(f"LLM raw score: {llm_raw_value:.3f}, soft-clamped: {clamped_value:.3f}")
            eval_record["final_value"] = clamped_value
            self.value_cache[prompt] = clamped_value
            return clamped_value, eval_record
        
        # 5. LLM Evaluation for 3+ numbers
        numbers_str = str(numbers)
        prompt = VALUE_PROMPT_CODEACT.format(input=numbers_str)
        
        if prompt in self.value_cache:
            self.stats['cache_hits'] += 1
            cached_value = self.value_cache[prompt]
            eval_record["from_cache"] = True
            eval_record["final_value"] = cached_value
            return cached_value, eval_record
        
        self.check_rate_limits()
        time.sleep(self.api_delay)
        self.stats['api_calls'] += self.n_evaluate_sample
        self.stats['daily_requests'] += self.n_evaluate_sample
        
        value_outputs = []
        for i in range(self.n_evaluate_sample):
            try:
                response = gemini_generate(prompt, n=1, temperature=0.0)[0]
                value_outputs.append(response.strip().lower())
            except:
                value_outputs.append("likely")
        
        # Score mapping: sure > likely > impossible (proper differentiation)
        # sure=1.0 (highest confidence), likely=0.6 (medium), impossible=0.001 (lowest)
        value_map = {'impossible': 0.001, 'likely': 0.6, 'sure': 1.0}
        value_names = []
        
        for output in value_outputs:
            last_word = output.strip().split()[-1] if output.strip() else "likely"
            value_names.append(last_word)
        
        # Store LLM judgments and reasoning
        eval_record["llm_judgments"] = value_outputs  # Raw responses
        eval_record["score_breakdown"] = {
            "raw_responses": value_outputs,
            "mapped_values": [value_map.get(name, 0.6) for name in value_names],
            "vote_counts": {name: value_names.count(name) for name in value_map.keys()}
        }
        
        # Add detailed reasoning
        eval_record["reasoning"].append(f"LLM evaluations: {value_names}")
        eval_record["reasoning"].append(f"Judgment votes: sure={value_names.count('sure')}, likely={value_names.count('likely')}, impossible={value_names.count('impossible')}")
        
        # Aggregate votes: average the mapped values (not sum which was problematic)
        raw_value = sum(value_map.get(name, 0.6) for name in value_names) / len(value_names) if value_names else 0.6
        boosted_value = raw_value * 1.2 if any(20 <= abs(n) <= 40 for n in numbers) and any(1 <= abs(n) <= 10 for n in numbers) else raw_value
        
        eval_record["reasoning"].append(f"Raw score: {raw_value:.3f}, Boosted score: {boosted_value:.3f}")
        eval_record["final_value"] = boosted_value
        self.value_cache[prompt] = boosted_value
        
        # NEW: Track evaluation for inconsistency detection (TIM Idea #4)
        if self.inconsistency_detector is not None:
            inconsistency_info = self.inconsistency_detector.record_evaluation(
                state=numbers,
                score=boosted_value,
                depth=getattr(self, 'current_depth', 0),
                reasoning=eval_record.get('reasoning', '')[-1] if eval_record.get('reasoning') else ''
            )
            if inconsistency_info and inconsistency_info.get('is_inconsistent'):
                eval_record['inconsistency_warning'] = {
                    'variance': inconsistency_info['variance'],
                    'scores': inconsistency_info['scores']
                }
        
        return boosted_value, eval_record
    
    def solve(self, numbers: List[int], verbose: bool = True) -> Tuple[List[str], 'TreeNode']:
        """COMPLETE BFS + BEAM SEARCH with ALL optimizations"""
        print(f"\n[SOLVE START] Input: {numbers}, verbose: {verbose}")
        
        TreeNode.node_counter = 0
        self.all_nodes = []
        self.solutions = []
        original_input = numbers.copy()
        self.original_input = original_input  # Store for export filename
        
        # Create root
        self.root = TreeNode(state=numbers, depth=0)
        self.all_nodes.append(self.root)
        
        # Track global seen states
        global_seen_states = set()
        global_seen_states.add(tuple(sorted(numbers)))
        
        queue = [(self.root, "")]
        
        for depth in range(self.max_steps):
            if verbose:
                print(f"\n{'='*70}")
                print(f"STEP {depth + 1}/{self.max_steps}")
                print(f"Current candidates: {len(queue)}")
            
            next_queue = []
            
            # === HYBRID SER MODE at DEPTH 0 ===
            if self.enable_ser and depth == 0:
                print(f"  [HYBRID SER] Depth 0: Using hybrid enumeration strategy")
                
                for node_idx, (node, history) in enumerate(queue):
                    if verbose:
                        print(f"\n  [SER] Processing root node: {node.state}")
                    
                    # Generate ALL ~24 first moves exhaustively
                    all_moves = self.generate_all_first_moves(node.state)
                    print(f"    [SER] Generated {len(all_moves)} exhaustive first moves")
                    
                    # Filter to PROMISING moves (up to n_select_sample) using basic heuristics
                    promising_moves = [m for m in all_moves if self.passes_basic_heuristics(m)]
                    promising_moves = promising_moves[:self.n_select_sample]
                    print(f"    [SER] Filtered to {len(promising_moves)} promising moves via heuristics")
                    
                    # LLM-EVALUATE only the promising moves
                    for move in promising_moves:
                        new_state = move['new_state']
                        state_tuple = tuple(sorted(new_state))
                        
                        if state_tuple in global_seen_states:
                            move['lm_evaluated'] = False
                            move['value'] = -float('inf')
                            continue
                        
                        # Check Dead-End Memory before LLM evaluation
                        if self.dead_end_memory is not None:
                            is_dead_end, matched_pattern = self.dead_end_memory.is_potential_dead_end(new_state)
                            if is_dead_end:
                                if verbose:
                                    print(f"      ⚠️  Skipping {new_state} - matches dead-end pattern")
                                move['lm_evaluated'] = False
                                move['value'] = -float('inf')
                                self.stats['deadend_memory_skipped'] += 1
                                continue
                        
                        # Evaluate via LLM
                        value, eval_record = self.evaluate_state(new_state, is_final=(len(new_state) == 1))
                        move['value'] = value
                        move['evaluation'] = eval_record
                        move['lm_evaluated'] = True
                        global_seen_states.add(state_tuple)
                        print(f"      [LLM] {new_state} → value: {value:.2f}")
                    
                    # HEURISTIC-SCORE the remaining moves
                    heuristic_moves = [m for m in all_moves if m not in promising_moves]
                    for move in heuristic_moves:
                        new_state = move['new_state']
                        state_tuple = tuple(sorted(new_state))
                        
                        if state_tuple in global_seen_states:
                            move['value'] = -float('inf')
                            move['lm_evaluated'] = False
                            continue
                        
                        # Check Dead-End Memory before heuristic scoring
                        if self.dead_end_memory is not None:
                            is_dead_end, matched_pattern = self.dead_end_memory.is_potential_dead_end(new_state)
                            if is_dead_end:
                                move['value'] = -float('inf')
                                move['lm_evaluated'] = False
                                self.stats['deadend_memory_skipped'] += 1
                                continue
                        
                        # Score via heuristic
                        heur_value = self.heuristic_score_move(move)
                        move['value'] = heur_value
                        move['lm_evaluated'] = False
                        global_seen_states.add(state_tuple)
                    
                    # SELECT with SEPARATE RANKING POOLS (Option 2 - uneven split)
                    # Split LLM and heuristic moves into separate pools
                    llm_moves = [m for m in all_moves if m.get('lm_evaluated', False) and m['value'] > -float('inf')]
                    heur_moves = [m for m in all_moves if not m.get('lm_evaluated', False) and m['value'] > -float('inf')]
                    
                    # Rank within each pool
                    llm_moves.sort(key=lambda x: x['value'], reverse=True)
                    heur_moves.sort(key=lambda x: x['value'], reverse=True)
                    
                    # Calculate uneven split: prioritize LLM (3 LLM + 2 heuristic for n=5)
                    llm_slots = max(1, self.n_select_sample // 2 + 1)  # For n=5: 3 LLM slots
                    heur_slots = max(0, self.n_select_sample - llm_slots)  # For n=5: 2 heuristic slots
                    
                    # Take top from each pool
                    selected_llm = llm_moves[:llm_slots]
                    selected_heur = heur_moves[:heur_slots]
                    selected_moves = selected_llm + selected_heur
                    
                    print(f"    [SER] Selected top {len(selected_moves)} moves from all {len(all_moves)}")
                    print(f"      LLM-evaluated: {len(selected_llm)} (from {len(llm_moves)} available)")
                    print(f"      Heuristic-scored: {len(selected_heur)} (from {len(heur_moves)} available)")
                    
                    # Create child nodes for selected moves
                    for move in selected_moves:
                        new_state = move['new_state']
                        state_tuple = tuple(sorted(new_state))
                        
                        step_desc = f"{move['thought']}\nResult: {move['observation']}"
                        new_history = history + "\n" + step_desc if history else step_desc
                        
                        child = TreeNode(
                            state=new_state,
                            parent=node,
                            action=move['action'],
                            depth=depth + 1
                        )
                        child.thought = move['thought']
                        child.code = move['code']
                        child.observation = move['observation']
                        child.path_history = new_history
                        child.value = move['value']
                        
                        if 'evaluation' in move:
                            child.evaluation = move['evaluation']
                        
                        # ADD: Link child to parent
                        node.children[child.id] = child
                        
                        next_queue.append((child, new_history))
                        self.all_nodes.append(child)
                    
                    # Set queue to the created depth-0 children for next iteration
                    queue = next_queue
            
            # === NORMAL LLM PROPOSALS MODE (depth >= 1) ===
            else:
                print(f"  [DEBUG] Entering LLM proposal mode for depth {depth}")
                print(f"  [DEBUG] Queue has {len(queue)} nodes to process")
                
                for node_idx, (node, history) in enumerate(queue):
                    print(f"\n  [DEBUG] Processing queue item {node_idx}: node.state={node.state}, len={len(node.state)}")
                    
                    if len(node.state) == 1:
                        print(f"  [DEBUG]   Skipping: state is single number")
                        continue
                    
                    if verbose:
                        print(f"\n  Node {node.id}: Generating proposals for {node.state}")
                    
                    proposals = self.propose(
                        node.state,
                        original_input=original_input,
                        history=history,
                        n_proposals=5,
                        current_depth=depth
                    )
                    
                    if verbose:
                        print(f"    → Generated {len(proposals)} unique proposals")
                    
                    for prop in proposals:
                        new_state = prop['new_state']
                        state_tuple = tuple(sorted(new_state))
                        
                        if state_tuple in global_seen_states:
                            continue
                        
                        # NEW: Check Dead-End Memory BEFORE creating child node
                        if self.dead_end_memory is not None:
                            is_dead_end, matched_pattern = self.dead_end_memory.is_potential_dead_end(new_state)
                            if is_dead_end:
                                if verbose:
                                    print(f"      ⚠️  Skipping {new_state} - matches dead-end pattern (reason: {matched_pattern.get('reason', 'unknown')})")
                                self.stats['deadend_memory_skipped'] += 1
                                continue
                        
                        global_seen_states.add(state_tuple)
                        
                        step_desc = f"{prop['thought']}\nResult: {prop['observation']}"
                        new_history = history + "\n" + step_desc if history else step_desc
                        
                        child = TreeNode(
                            state=new_state,
                            parent=node,
                            action=prop['action'],
                            depth=depth + 1
                        )
                        child.thought = prop['thought']
                        child.code = prop['code']
                        child.observation = prop['observation']
                        child.path_history = new_history
                        
                        # ADD: Link child to parent
                        node.children[child.id] = child
                        
                        next_queue.append((child, new_history))
                        self.all_nodes.append(child)
            
            print(f"  [DEBUG] next_queue size: {len(next_queue)}")
            
            if not next_queue:
                if verbose:
                    print("\n  ⚠ No new proposals. Stopping.")
                break
            
            # Evaluate all new nodes (pass 1: cheap)
            if verbose:
                print(f"\n  Evaluating {len(next_queue)} new states...")
            
            for child, _ in next_queue:
                value, eval_record = self.evaluate_state(child.state, is_final=(len(child.state) == 1), force_llm=False)
                child.value = value
                child.evaluation = eval_record
            
            # Optional: upgrade top-M 2-number states with LLM scoring + soft clamp
            two_number_candidates = [child for child, _ in next_queue if len(child.state) == 2]
            if two_number_candidates:
                m = 2 * self.n_select_sample
                two_number_candidates.sort(key=lambda c: c.value, reverse=True)
                shortlist = set(two_number_candidates[:m])
                
                if verbose:
                    print(f"\n  [2-NUMBER LLM] Upgrading top {len(shortlist)}/{len(two_number_candidates)} two-number states with LLM + soft clamp...")
                
                for child, _ in next_queue:
                    if child in shortlist:
                        value, eval_record = self.evaluate_state(child.state, is_final=False, force_llm=True)
                        child.value = value
                        child.evaluation = eval_record
            
            # ...existing code that uses eval_record['heuristic_checks']['two_number_reaches_24']
            # to create the ONE final child, and then sorts next_queue and selects top-K...
        
        # Check for solutions
        for node in self.all_nodes:
            if len(node.state) == 1 and abs(node.state[0] - 24) < 0.001:
                node.is_solution = True
                self.solutions.append(node)
        
        self.stats['total_nodes'] = len(self.all_nodes)
        self.stats['solutions_found'] = len(self.solutions)
        
        # NEW: Add Dead-End Memory stats
        if self.dead_end_memory is not None:
            self.stats['deadend_memory'] = self.dead_end_memory.get_stats()
            self.stats['deadend_memory_skipped'] = self.dead_end_memory.total_skipped
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"✓ Found {len(self.solutions)} solution(s)")
            print(f"  Total nodes: {self.stats['total_nodes']}")
            print(f"  API calls: {self.stats['api_calls']}")
            print(f"  Cache hits: {self.stats['cache_hits']}")
            if self.dead_end_memory is not None:
                mem_stats = self.dead_end_memory.get_stats()
                print(f"  Dead-End Memory:")
                print(f"    - Patterns stored: {mem_stats['patterns_stored']}")
                print(f"    - States checked: {mem_stats['total_checked']}")
                print(f"    - States skipped: {mem_stats['total_skipped']} ({mem_stats['skip_rate']})")
                api_saved = mem_stats['total_skipped'] * self.n_evaluate_sample
                print(f"    - API calls saved: {api_saved}")
        
        print(f"[SOLVE END]\n")
        return [self.reconstruct_solution_path(node) for node in self.solutions], self.root
    
    def reconstruct_solution_path(self, node: 'TreeNode') -> str:
        """Reconstruct the solution path from root to node"""
        path = []
        current = node
        
        while current.parent is not None:
            step_info = f"Thought: {current.thought}\nCode:\n{current.code}\nResult: {current.observation}"
            path.append(step_info)
            current = current.parent
        
        path.reverse()
        return "\n\n".join(path)
    
    def export_tree_to_json(self, filename: str = None) -> str:
        """Export the entire search tree to JSON"""
        import os
        
        # Create raw_tree folder if it doesn't exist
        raw_tree_folder = "raw_tree"
        if not os.path.exists(raw_tree_folder):
            os.makedirs(raw_tree_folder)
            print(f"✓ Created folder: {raw_tree_folder}")
        
        if filename is None:
            # Include the problem in the filename
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            if hasattr(self, 'original_input') and self.original_input:
                problem_str = "_".join(str(n) for n in self.original_input)
                filename = f"{raw_tree_folder}/game24_tree_{problem_str}_{timestamp}.json"
            else:
                filename = f"{raw_tree_folder}/game24_tree_{timestamp}.json"
        
        stats_serializable = self.stats.copy()
        if 'session_start' in stats_serializable:
            try:
                stats_serializable['session_start'] = stats_serializable['session_start'].isoformat()
            except AttributeError:
                pass
        
        # NEW: Add Dead-End Memory stats to export
        mode = 'CodeAct with Dead-End Memory' if self.enable_deadend_memory else 'CodeAct'
        
        tree_data = {
            'metadata': {
                'timestamp': datetime.now().isoformat(),
                'mode': mode,
                'parameters': {
                    'temperature': self.temperature,
                    'n_evaluate_sample': self.n_evaluate_sample,
                    'n_select_sample': self.n_select_sample,
                    'max_steps': self.max_steps,
                    'api_delay': self.api_delay,
                    'enable_deadend_memory': self.enable_deadend_memory
                },
                'statistics': stats_serializable,
                'deadend_memory_summary': {
                    'enabled': self.enable_deadend_memory,
                    'patterns_stored': len(self.dead_end_memory.patterns) if self.dead_end_memory else 0,
                    'total_checked': self.dead_end_memory.total_checked if self.dead_end_memory else 0,
                    'total_skipped': self.dead_end_memory.total_skipped if self.dead_end_memory else 0,
                    'api_calls_saved': (self.dead_end_memory.total_skipped * self.n_evaluate_sample) if self.dead_end_memory else 0
                }
            },
            'nodes': [node.to_dict() for node in self.all_nodes],
            'solutions': [node.id for node in self.solutions],
            # NEW: Add Inconsistency Detection Report (TiM Idea #4)
            'inconsistency_report': {
                'enabled': self.inconsistency_detector is not None,
                'statistics': self.inconsistency_detector.get_stats() if self.inconsistency_detector else {},
                'inconsistent_states': self.inconsistency_detector.get_inconsistent_states() if self.inconsistency_detector else [],
                'suspicious_states': self.inconsistency_detector.get_suspicious_states() if self.inconsistency_detector else []
            }
        }
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(tree_data, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Tree exported to: {filename}")
        if self.enable_deadend_memory and self.dead_end_memory:
            print(f"  • Dead-End Memory: {self.dead_end_memory.total_skipped} states skipped")
            print(f"  • API calls saved: {self.dead_end_memory.total_skipped * self.n_evaluate_sample}")
        return filename

print("✓ Game24TreeOfThoughts COMPLETE solver class loaded")
print("✓ Includes: SER, exhaustive_depth1, DFP, global tracking, hybrid evaluation, Dead-End Memory")

In [8]:
# ============================================================================
# STEP 7: VISUALIZATION & ANALYSIS FUNCTIONS
# ============================================================================

def visualize_tree_codeact(root_node, max_depth=3):
    """Visualize the CodeAct search tree structure"""
    def print_node(node, depth=0, prefix="", is_last=True):
        if depth > max_depth:
            return
        
        branch = "└── " if is_last else "├── "
        extension = "    " if is_last else "│   "
        
        if node.depth == 0:
            line = f"🌳 ROOT: {node.state or 'Start'}"
        else:
            action = node.action[:50] + "..." if len(node.action) > 50 else node.action
            value_str = f"(value: {node.value:.2f})" if node.value > 0 else ""
            solution_marker = " ✅ SOLUTION" if node.is_solution else ""
            pruned_marker = " ❌ PRUNED" if node.is_pruned else ""
            
            line = f"{branch}[D{node.depth}] {action} {value_str}{solution_marker}{pruned_marker}"
        
        print(prefix + line)
        
        children = list(node.children.values())
        for i, child in enumerate(children):
            is_last_child = (i == len(children) - 1)
            new_prefix = prefix + extension
            print_node(child, depth + 1, new_prefix, is_last_child)
    
    print("\n" + "="*70)
    print("🎄 SEARCH TREE VISUALIZATION")
    print("="*70)
    print_node(root_node)
    print("="*70 + "\n")

def analyze_tree(root_node):
    """Analyze and display statistics about the search tree"""
    stats = {
        'total_nodes': 0,
        'by_depth': {},
        'solutions': 0,
        'pruned': 0,
        'max_depth': 0,
        'avg_branching': 0,
        'total_children': 0
    }
    
    def traverse(node):
        stats['total_nodes'] += 1
        depth = node.depth
        
        if depth not in stats['by_depth']:
            stats['by_depth'][depth] = 0
        stats['by_depth'][depth] += 1
        
        if depth > stats['max_depth']:
            stats['max_depth'] = depth
        
        if node.is_solution:
            stats['solutions'] += 1
        if node.is_pruned:
            stats['pruned'] += 1
        
        num_children = len(node.children)
        if num_children > 0:
            stats['total_children'] += num_children
        
        for child in node.children.values():
            traverse(child)
    
    traverse(root_node)
    
    internal_nodes = stats['total_nodes'] - stats['by_depth'].get(stats['max_depth'], 0)
    stats['avg_branching'] = stats['total_children'] / internal_nodes if internal_nodes > 0 else 0
    
    print("\n" + "="*70)
    print("📊 TREE ANALYSIS")
    print("="*70)
    print(f"Total Nodes:        {stats['total_nodes']}")
    print(f"Max Depth:          {stats['max_depth']}")
    print(f"Solutions Found:    {stats['solutions']}")
    print(f"Pruned Nodes:       {stats['pruned']}")
    print(f"Avg Branching:      {stats['avg_branching']:.2f}")
    print()
    print("Nodes by Depth:")
    for depth in sorted(stats['by_depth'].keys()):
        count = stats['by_depth'][depth]
        bar = "█" * min(count, 50)
        print(f"  Depth {depth}: {count:3d} {bar}")
    print("="*70 + "\n")

print("✓ Visualization and analysis functions loaded")

---

## 🎯 Ready to Test!

You've completed the setup. Now you can:

1. **Create a solver instance** (see cells below)
2. **Test with example puzzles**
3. **View results and tree analysis**

---

In [9]:
# ============================================================================
# EXAMPLE: Solve Game of 24 puzzle [1, 2, 4, 7]
# ============================================================================
# Solution: (7-2+1)*4 = 6*4 = 24

input_numbers = [4,6, 7, 10]

print("="*70)
print("🎮 GAME OF 24 SOLVER - TEST PUZZLE")
print("="*70)
print(f"\n📊 Input Numbers: {input_numbers}")
print(f"🎯 Goal: Use +, -, *, / to make 24")
print(f"💡 Solution: (7-2+1)*4 = 24")
print(f"\n⚙️  Solver Configuration:")
print(f"   • Temperature: 0.7 (balanced exploration)")
print(f"   • Evaluation samples: 3 (per state)")
print(f"   • Beam width: 10 (keep top 10 candidates per level)")
print(f"   • Max search depth: 6 (up to 6 operations)")
print(f"   • Rate limit: 3.5s between API calls")
print(f"   • Exhaustive first moves: OFF (use LLM proposals)")
print(f"   • SER (Selective Exhaustive Rescue): OFF")
print(f"   • DFP (Delayed Fraction Preservation): ON")
print()

# Create solver instance
solver = Game24TreeOfThoughts(
    temperature=0.7,
    n_evaluate_sample=3,
    n_select_sample=5,
    max_steps=6,
    api_delay=API_DELAY,
    selection_method='greedy',
    exhaustive_depth1=False,
    enable_ser=False,
    enable_deadend_memory=True
)

print(f"{'='*70}")
print(f"🚀 STARTING SEARCH...")
print(f"{'='*70}\n")

# Run the solver
solutions, root = solver.solve(input_numbers, verbose=True)

# Display results
print(f"\n{'='*70}")
print(f"RESULTS")
print(f"{'='*70}\n")

if solutions:
    print(f"✅ SUCCESS! Found {len(solutions)} solution(s)\n")
    for i, sol in enumerate(solutions, 1):
        print(f"Solution {i}:")
        print("-" * 70)
        print(sol)
        print("-" * 70)
        print()
else:
    print(f"❌ No solution found in {solver.stats['total_nodes']} nodes explored")
    print(f"\n💡 Try enabling SER or exhaustive_depth1 for harder puzzles:")
    print(f"   • exhaustive_depth1=True: Try ALL ~24 possible first moves")
    print(f"   • enable_ser=True: Rescue with exhaustive search if LLM proposals are weak")
    print()

# Display statistics
print(f"{'='*70}")
print(f"📊 STATISTICS")
print(f"{'='*70}\n")
print(f"Total nodes explored:   {solver.stats['total_nodes']}")
print(f"API calls made:         {solver.stats['api_calls']}")
print(f"Cache hits:             {solver.stats['cache_hits']}")
print(f"Code executions:        {solver.stats['code_executions']}")
print(f"Code errors:            {solver.stats['code_errors']}")
print(f"Daily requests used:    {solver.stats['daily_requests']}/14000")
print()

# Export tree to JSON
json_file = solver.export_tree_to_json()
print(f"\n📁 Complete search tree exported to: {json_file}")
print(f"\n💾 You can analyze this JSON to understand the search process:")
print(f"   • All nodes visited")
print(f"   • Parent-child relationships")
print(f"   • State evaluations and reasoning")
print(f"   • Solution paths")
print()

print(f"{'='*70}")
if solutions:
    print(f"✨ PUZZLE SOLVED! ✨")
else:
    print(f"🔧 Consider tweaking parameters for harder puzzles")
print(f"{'='*70}")

## Testing Balanced Evaluation Prompt with Gemma 3.1B

Test both current and balanced VALUE_PROMPT versions to see if Gemma can follow concise format and compare output quality.

In [36]:
# Current VALUE_PROMPT (verbose)
CURRENT_VALUE_PROMPT = """<start_of_turn>user
Can these numbers reach 24 using +, -, *, /?

Evaluate BRIEFLY then answer with ONE WORD on a new line.

RULES:
1. Two numbers + operation ≈ 24 (result within ±0.01)? → "sure"
2. Direct path visible with 3+ numbers (e.g., a*b+c≈24 or a*b-c≈24)? → "sure"
3. Plausible path exists but requires multiple steps/careful ordering? → "likely"
4. No sequence of operations can reach 24? → "impossible"

TOLERANCE: Accept floating-point results within 0.01 of 24 (e.g., 23.99-24.01).

Numbers: {input}
Brief check:
<end_of_turn>
<start_of_turn>model
"""

# Simplified BALANCED prompt - minimal structure, explicit about brevity
BALANCED_VALUE_PROMPT = '''<start_of_turn>user
Numbers: {input}

Can you make 24? Answer ONLY with: sure, likely, or impossible

sure = reaches 24 exactly or has direct path
likely = solvable but needs multiple steps
impossible = cannot reach 24
<end_of_turn>
<start_of_turn>model
'''

print("✓ Prompts loaded for comparison testing")
print(f"  Current prompt length: {len(CURRENT_VALUE_PROMPT)} chars")
print(f"  Balanced prompt length: {len(BALANCED_VALUE_PROMPT)} chars")
print(f"  Space saving: {100 * (1 - len(BALANCED_VALUE_PROMPT)/len(CURRENT_VALUE_PROMPT)):.1f}%")

In [33]:
# Test cases from the actual puzzle [6, 9, 9, 10] that worked
test_states = [
    # From the successful path that found solution
    ([1.5, 9, 10], "should be 'sure' - has direct path 1.5*10+9=24"),
    ([15.0, 9], "should be 'sure' - 15+9=24"),
    ([13.5, 10], "should be 'impossible' - no operation reaches 24"),
    ([19, 1.5], "should be 'likely' - close but no direct path"),
    
    # Other cases to verify
    ([12, 12], "should be 'sure' - 12+12=24"),
    ([8, 3], "should be 'sure' - 8*3=24"),
    ([20, 4], "should be 'likely' - 20+4=24 exact!"),
    ([2, 5, 7], "should be 'sure' - 2*5*7=70? no. (5+7)*2=24 yes"),
    ([100, 0.5, 3], "should be 'impossible' - intractable ratio"),
]

print("=" * 70)
print("TEST STATES FROM ACTUAL PUZZLE")
print("=" * 70)
for state, expected in test_states:
    state_str = " ".join(f"{x:.1f}" if isinstance(x, float) else str(x) for x in state)
    print(f"\n{state_str:20} → {expected}")

In [34]:
import time
from pathlib import Path

async def test_prompt_version(state_list, prompt_template, prompt_name, model_name="gemini-2.0-flash-exp", output_file=None):
    """Test a prompt template against test states and write results to file"""
    
    # Setup output file
    if output_file is None:
        output_file = WORKSPACE_DIR / f"prompt_test_{prompt_name.replace(' ', '_')}.txt"
    
    output_lines = []
    
    def write_out(msg=""):
        """Print and append to output list"""
        print(msg)
        output_lines.append(msg)
    
    write_out("=" * 70)
    write_out(f"TESTING: {prompt_name}")
    write_out(f"Model: {model_name}")
    write_out(f"Timestamp: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    write_out("=" * 70)
    
    results = []
    total_length = 0
    
    for state, expected in state_list:
        state_str = " ".join(f"{x:.1f}" if isinstance(x, float) else str(x) for x in state)
        
        # Format the prompt
        prompt = prompt_template.format(input=state_str)
        
        try:
            # Call the model
            response = model.generate_content(prompt)
            output = response.text.strip()
            
            # Clean output to get just the judgment
            lines = output.split('\n')
            judgment = lines[0].lower().strip() if lines else "error"
            
            # Extract key judgment word
            for word in ['sure', 'likely', 'impossible']:
                if word in judgment:
                    judgment = word
                    break
            
            output_length = len(output)
            total_length += output_length
            
            # Store result
            result = {
                'state': state_str,
                'judgment': judgment,
                'length': output_length,
                'expected': expected,
                'full_output': output[:100] + "..." if len(output) > 100 else output
            }
            results.append(result)
            
            write_out(f"\n{state_str:20} → {judgment:12} ({output_length:3} chars)")
            write_out(f"  Expected: {expected}")
            if output_length < 50:
                write_out(f"  Output: {output}")
            
            time.sleep(0.5)  # Rate limiting
            
        except Exception as e:
            write_out(f"\n{state_str:20} → ERROR: {str(e)[:50]}")
            results.append({
                'state': state_str,
                'error': str(e)
            })
            time.sleep(0.5)
    
    # Summary
    successful = len([r for r in results if 'error' not in r])
    if successful > 0:
        avg_length = total_length / successful
    else:
        avg_length = 0
    
    write_out("\n" + "=" * 70)
    write_out(f"SUMMARY: {prompt_name}")
    write_out("=" * 70)
    write_out(f"Successful responses: {successful}/{len(results)}")
    write_out(f"Average output length: {avg_length:.0f} chars")
    write_out(f"Total prompt size: {len(prompt_template)} chars")
    write_out(f"Total output size: {total_length} chars")
    write_out(f"Prompt template:\n{prompt_template[:200]}...\n")
    
    # Write to file
    with open(output_file, 'w') as f:
        f.write('\n'.join(output_lines))
    
    write_out(f"\n✓ Results written to: {output_file}")
    
    return results, output_lines

# Run comparison
print("\nPreparing to test both prompts...")
print(f"Number of test states: {len(test_states)}")
print(f"This will make {len(test_states) * 2} API calls (about 2 mins with rate limiting)")
print(f"Results will be saved to {WORKSPACE_DIR}")


In [37]:
# Run the tests
print("\n" + "="*70)
print("PROMPT COMPARISON TEST - STARTING")
print("="*70)

# Test current prompt
print("\n[1/2] Testing CURRENT prompt...")
current_results, current_output = await test_prompt_version(
    test_states, 
    CURRENT_VALUE_PROMPT, 
    "CURRENT_VERBOSE_PROMPT",
    output_file=WORKSPACE_DIR / "PROMPT_TEST_CURRENT.txt"
)

# Test balanced prompt
print("\n[2/2] Testing BALANCED prompt...")
balanced_results, balanced_output = await test_prompt_version(
    test_states, 
    BALANCED_VALUE_PROMPT, 
    "BALANCED_CONCISE_PROMPT",
    output_file=WORKSPACE_DIR / "PROMPT_TEST_BALANCED.txt"
)

# Create comparison report
comparison_report = []
comparison_report.append("=" * 70)
comparison_report.append("PROMPT COMPARISON REPORT")
comparison_report.append("=" * 70)
comparison_report.append("")
comparison_report.append(f"Test date: {time.strftime('%Y-%m-%d %H:%M:%S')}")
comparison_report.append(f"Test states: {len(test_states)}")
comparison_report.append("")

# Metrics
current_successful = len([r for r in current_results if 'error' not in r])
balanced_successful = len([r for r in balanced_results if 'error' not in r])

current_avg_len = sum(r.get('length', 0) for r in current_results if 'error' not in r) / max(1, current_successful)
balanced_avg_len = sum(r.get('length', 0) for r in balanced_results if 'error' not in r) / max(1, balanced_successful)

comparison_report.append("METRICS COMPARISON")
comparison_report.append("-" * 70)
comparison_report.append(f"\nCURRENT PROMPT:")
comparison_report.append(f"  Size: {len(CURRENT_VALUE_PROMPT)} chars")
comparison_report.append(f"  Successful: {current_successful}/{len(test_states)}")
comparison_report.append(f"  Avg response length: {current_avg_len:.0f} chars")

comparison_report.append(f"\nBALANCED PROMPT:")
comparison_report.append(f"  Size: {len(BALANCED_VALUE_PROMPT)} chars")
comparison_report.append(f"  Successful: {balanced_successful}/{len(test_states)}")
comparison_report.append(f"  Avg response length: {balanced_avg_len:.0f} chars")

comparison_report.append(f"\nIMPROVEMENTS:")
prompt_reduction = 100 * (1 - len(BALANCED_VALUE_PROMPT) / len(CURRENT_VALUE_PROMPT))
response_reduction = 100 * (1 - balanced_avg_len / current_avg_len)
comparison_report.append(f"  Prompt size reduction: {prompt_reduction:.1f}%")
comparison_report.append(f"  Avg response reduction: {response_reduction:.1f}%")

# Write comparison
report_file = WORKSPACE_DIR / "PROMPT_COMPARISON_REPORT.txt"
with open(report_file, 'w') as f:
    f.write('\n'.join(comparison_report))

print("\n" + '\n'.join(comparison_report))
print(f"\n✓ Comparison report written to: {report_file}")


In [14]:
# ...existing code above...
# Create ONE child with the closest result
if best_result is not None:
    if verbose:
        print(f"      ✅ Best operation: {best_op_str} = {best_result} (distance: {best_distance:.3f})")
    
    new_state = [best_result]
    state_tuple = tuple(sorted(new_state))
    
    # IMPORTANT: Don't let global_seen_states suppress creating a final [24] node.
    # Otherwise, 2 different parents that can both reach 24 will collapse into 1 recorded solution.
    is_final_24 = abs(best_result - 24) < 0.001
    should_create = is_final_24 or (state_tuple not in global_seen_states)
    
    if should_create:
        code = f"numbers = {list(child.state)}\n"
        code += f"a, b = {a}, {b}\n"
        code += best_op_code + "\n"
        code += "new_numbers = [result]\n"
        
        step_desc = f"Operation: {best_op_str}\nResult: {new_state}"
        new_history = child.path_history + "\n" + step_desc if child.path_history else step_desc
        
        solution_node = TreeNode(
            state=new_state,
            parent=child,
            action=best_op_str,
            depth=depth + 2,
        )
        solution_node.thought = f"2-number operation: {best_op_str} = {best_result}"
        solution_node.code = code
        solution_node.observation = str(new_state)
        solution_node.path_history = new_history
        
        # Mark as solution if it's exactly 24
        if is_final_24:
            solution_node.is_solution = True
            solution_node.value = 1.0
            if verbose:
                print(f"      ✅ SOLUTION FOUND: {new_state[0]}")
        else:
            solution_node.value = 0.5  # Good intermediate result
            if verbose:
                print(f"      ℹ️  Intermediate result: {new_state[0]}")
        
        # Link to parent
        child.children[solution_node.id] = solution_node
        
        # Keep global_seen_states for non-final nodes, to reduce loops as before
        if not is_final_24:
            global_seen_states.add(state_tuple)
        self.all_nodes.append(solution_node)
# ...existing code below...

In [18]:
print("=" * 100)
print("PROPOSAL SORTING & SELECTION AT DEPTH 1")
print("=" * 100)

print("""
QUESTION: When proposals are made at depth 1, are they sorted as a WHOLE?
          (e.g., 5 parent nodes × 5 proposals each = 25 total, sorted together?)
          OR sorted WITHIN EACH PARENT?

ANSWER: ✅ SORTED AS A WHOLE - ALL proposals together!

FLOW AT DEPTH 1:
━━━━━━━━━━━━━━━━━

1️⃣  START OF ITERATION:
    queue = [Node1, Node2, Node3, Node4, Node5]  (from depth 0 selection)
    next_queue = []  (empty)
    
2️⃣  LOOP THROUGH EACH PARENT NODE:
    for node_idx, (node, history) in enumerate(queue):  # Line 692
        proposals = self.propose(node.state, ...)  # 5 proposals per parent
        
        # Create children for each proposal and ADD TO next_queue
        for prop in proposals:
            child = TreeNode(...)
            next_queue.append((child, new_history))  # Add all children here
    
    After loop: next_queue has ALL children from ALL parents
                Example: 5 parents × 5 proposals = up to 25 nodes
    
3️⃣  EVALUATE ALL IN next_queue:
    for child, _ in next_queue:                    # Line 762 (approx)
        value, eval_record = self.evaluate_state(child.state)
        child.value = value
        child.evaluation = eval_record
    
    Now all 25 children have scores assigned
    
4️⃣  SORT ALL TOGETHER (GLOBAL SORT):
    next_queue.sort(key=lambda x: x[0].value, reverse=True)  # Line 845
    
    ⚠️  This sorts ALL children from ALL parents together!
    Not parent-by-parent, but GLOBALLY across all proposals
    
5️⃣  SELECT TOP-K:
    selected = next_queue[:self.n_select_sample]  # Line 846 (keep top 5)
    queue = selected                              # Line 856
    
    Only the top-5 scoring children advance to next depth

EXAMPLE WITH [4,5,7,9] PUZZLE:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DEPTH 0→1 Creates 5 nodes at depth 1:
  Child of Root
    ├─ Node 1: [36,5,7] value=1.04 ✅ "sure" (2/3 votes)
    ├─ Node 2: [28,5,9] value=1.20 ✅ "sure" (3/3 votes)  ← HIGHEST!
    ├─ Node 3: [4,4,7]  value=1.00 ✅ "sure" (3/3 votes)
    ├─ Node 4: [3,5,9]  value=0.733 "likely"
    └─ Node 5: [12,4,9] value=0.867 ✅ "sure" (2/3 votes)

DEPTH 1→2 (Each depth-1 node generates 5 proposals):
  
  Proposals from Node 1 ([36,5,7]):
    1.1: [31,7]  value=1.0 (reached 24!)
    1.2: [29,5]  value=1.0 (reached 24!)
    1.3: [12,36] value=1.0 (reached 24!)
    1.4: [35,36] value=0.001 (can't reach 24)
    1.5: [7.2,7] value=0.001 (can't reach 24)
  
  Proposals from Node 2 ([28,5,9]):
    2.1: [...] value=?
    2.2: [...] value=?
    ...
  
  Proposals from Nodes 3-5:
    3.1, 3.2, ..., 5.5: [all added to next_queue]
  
  TOTAL in next_queue: ~25 nodes (could be less due to deduplication)
  
  GLOBAL SORT by value (all 25 together):
    Sorted by descending value score
    
  SELECT TOP-5:
    Take the best 5 from all 25 proposals
    
  ❌ ISSUE: Only Node 1's proposals were explored in practice
          Even though Node 2 (value 1.20) > Node 1 (value 1.04)
          This suggests: early termination or selection bug

KEY INSIGHTS:
━━━━━━━━━━━━━

✅ Sorting is GLOBAL (all proposals together, not per-parent)
✅ Selection uses value score, not judgment (sure/likely/impossible)
✅ Top-5 scoring children are selected for next depth
❌ But in practice, only Node 1's branch was explored
❌ Suggests: Either Node 1's children had higher values than others,
           OR there's early termination when solution is found,
           OR sorting is working backward (reverse=False instead of True)
""")

print("\nTo debug this, add logging BEFORE line 845:")
print("""
print(f"\\n[DEBUG] Before sorting (first 10 nodes):")
for i in range(min(10, len(next_queue))):
    node, _ = next_queue[i]
    print(f"  {i+1}. ID {node.id}: parent={node.parent.id}, value={node.value:.3f}, state={node.state}")
""")
